# FAITH-Detect — Full experimental grid (Colab T4)

Runs the **same** `run_full_experiment` code as the local smoke run, with a heavier config
(5 seeds, more epochs, full RAID OOD + attacks). The local smoke is therefore a faithful
preview of these results.

**Steps:** check GPU → install deps → get code + MAiDE-up CSV → run grid → figures → download.

> Set Runtime → Change runtime type → **T4 GPU** before running.

In [ ]:
!nvidia-smi -L
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())

In [ ]:
# 1) Dependencies (Colab already has torch/transformers; add the rest).
!pip -q install captum statsmodels umap-learn shap lime datasets spacy >/dev/null
!python -m spacy download en_core_web_sm -q
import nltk
for pkg in ['stopwords','punkt','punkt_tab','wordnet','omw-1.4','averaged_perceptron_tagger_eng']:
    nltk.download(pkg, quiet=True)
print('deps ready')

In [ ]:
# 2) Get the FAITH-Detect code.
#    Option A: clone your repo (replace URL):
# !git clone https://github.com/<you>/FAITH-Detect.git
#    Option B: upload a zip of the repo and unzip:
from google.colab import files
import os, zipfile
if not os.path.exists('FAITH-Detect'):
    print('Upload FAITH-Detect.zip (the project folder zipped):')
    up = files.upload()
    name = next(iter(up))
    with zipfile.ZipFile(name) as z: z.extractall('.')
%cd FAITH-Detect
import sys; sys.path.insert(0, 'src')

In [ ]:
# 3) Provide the MAiDE-up CSV (all_data.csv). Upload it to ./data/all_data.csv
import os
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/all_data.csv'):
    from google.colab import files
    print('Upload all_data.csv:')
    up = files.upload()
    import shutil; shutil.move(next(iter(up)), 'data/all_data.csv')
print('CSV ready:', os.path.exists('data/all_data.csv'))

In [ ]:
# 4) Configure and run the full grid.
from faithdetect.experiment import ExperimentConfig, run_full_experiment
from faithdetect.utils.logging import save_json
from faithdetect.viz import make_all_figures

cfg = ExperimentConfig(
    name='full',
    data_csv='data/all_data.csv',
    encoder_name='roberta-base',         # try 'roberta-large' for an ablation
    variants=('baseline','hardmask','softreg'),
    seeds=(0,1,2,3,4),
    epochs=5, batch_size=32, max_length=256,
    # Cross-domain: 'abstracts' is cheap to stream. To include RAID 'reviews' (same task),
    # raise ood_max_scan a lot (the CSV is domain-sorted) or pre-download (see cell 5).
    ood_domains=('abstracts',),
    ood_cap_per_group=200, ood_max_scan=120000,
    attack_types=('function_word','synonym','whitespace'), attack_rate=0.5,
    faithfulness_n_texts=100, ig_steps=50, n_example_explanations=8,
    device='cuda',
)
results = run_full_experiment(cfg)
save_json('results/full_results.json', results)
figs = make_all_figures(results, 'figures')
print('done:', len(figs), 'figures')

In [ ]:
# 5) (Optional) Target RAID 'reviews' (cross-domain, same task) + cross-generator by
#    pre-downloading the CSV and filtering with pandas chunks (faster than deep streaming).
#    Then point load_raid_sample at the local file via a small monkeypatch, or just build a
#    frame here and evaluate. Skeleton:
# from huggingface_hub import hf_hub_download
# import pandas as pd
# path = hf_hub_download('liamdugan/raid','train.csv',repo_type='dataset')
# keep=[]
# for ch in pd.read_csv(path, chunksize=200000):
#     sub = ch[(ch['domain']=='reviews') & (ch['attack']=='none')]
#     keep.append(sub)
#     if sum(len(k) for k in keep) > 8000: break
# reviews = pd.concat(keep)
# reviews['label'] = (reviews['model']!='human').astype(int)
# reviews['text'] = reviews['generation']
# print(reviews['model'].value_counts())

In [ ]:
# 6) Zip and download results + figures.
!zip -qr faithdetect_outputs.zip results figures
from google.colab import files; files.download('faithdetect_outputs.zip')